In [1]:
"""
Sales Analytics Without Pandas

Tasks 1-7 use only the standard library. pandas is allowed in task 8, to check your work.
"""
import csv
from pathlib import Path
import re

import pandas as pd
from pathlib import Path


In [2]:
# importing data from CSV file

DATA_PATH = Path("data") / "electronics_sales.csv"

def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    try:
        with path.open(newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            return list(reader)
    except:
        raise FileNotFoundError("File not found! ")
    
# testing function
rows = read_sales()
print(len(rows))
rows[:3]

50


[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'}]

---

In [3]:
def parse_number(value: str) -> float | None:
    """Convert a messy numeric string to a float, or return None if it cannot be trusted."""
    # convert quantity and unit_price_kes 
    if value is None:
        return None
    # cleaned = value.strip().strip("\"'").upper().replace("KES", "").replace(",", "").strip()
    
    # remove all non numeric characters
    # strip to remove whitespaces
    # remove single/ double quotes
    cleaned = re.sub(r"[^0-9.]", "", value.strip().strip("\"'"))
    try:
        return float(cleaned)
    except ValueError:
        return None


In [4]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return (clean_rows, skipped_rows). Clean rows have numeric quantity, unit price and revenue."""
    clean_rows, skipped_rows = [], []

    # starting enum at line 2 since line 1 is the header line in the data
    for line, row in enumerate(rows, start = 2):
        qty = parse_number(row["quantity"])
        price = parse_number(row["unit_price_kes"])

        if qty is None:
            skipped_rows.append({"Line": line, "sale_id": row["sale_id"], "reason": f"Invalid quantiry: {row['quantity']!r}"})
        elif price is None:
            skipped_rows.append({"line": line, "sale_id": row["sale_id"], "reason": f"invalid unit_price_kes: {row['unit_price_kes']!r}"})
        else:
            clean_rows.append({**row, "quantity": int(qty), "unit_price_kes": price})

    return clean_rows, skipped_rows


# testing celan vs skipped rows
clean, skipped = convert_rows(rows)

print(len(clean)) # 46 clean rows
skipped


46


[{'Line': 7, 'sale_id': 'S-3006', 'reason': "Invalid quantiry: 'two'"},
 {'line': 31, 'sale_id': 'S-3030', 'reason': "invalid unit_price_kes: 'N/A'"},
 {'Line': 36, 'sale_id': 'S-3035', 'reason': "Invalid quantiry: ''"},
 {'line': 40, 'sale_id': 'S-3039', 'reason': "invalid unit_price_kes: ''"}]

---

In [5]:
def total_revenue(rows: list[dict]) -> float:
    # loop  through each row
    # get total for each row, 
    # add total using sum
    return sum(row["quantity"] * row["unit_price_kes"] for row in rows)

total = total_revenue(clean)
# output tota rev in 2 decimal places
print(f"Total Revenue {total:,.2f}")

Total Revenue 1,310,812.00


In [6]:

def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Total revenue grouped by any column (product, store, payment_method...)."""
    totals = {}

    # getting total for every row
    for row in rows:
        key = row[field]
        revenue = row["quantity"] * row["unit_price_kes"]

        # start a new total for every new key found
        if key in totals:
            totals[key] += revenue
        else:
            totals[key] = revenue
    return totals


In [7]:
# testing revenue by function
by_product = revenue_by(clean, "product")
by_store = revenue_by(clean, "store")
by_payment = revenue_by(clean, "payment_method")

for key, revenue in sorted(by_store.items(), key=lambda item: item[1], reverse=True):
    print(f"{key:<20}   : {revenue:,.2f}")


Kisumu Mega Plaza      : 449,678.00
Nairobi CBD            : 331,085.00
Westgate               : 235,979.00
Mombasa City Mall      : 154,082.00
Nakuru Westside        : 139,988.00


In [8]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the n highest (key, value) pairs, highest first."""
    # using -item[1] to sort by revenue first
    # incase of tie use item[0] to sort alphabetically
    ranked = sorted(totals.items(), key=lambda item: (-item[1], item[0]))
    return ranked[:n]


for rank, (key, revenue) in enumerate(top_n(by_product, 3), start=1):
    print(f"{rank}. {key:<30} : {revenue}")

1. HP 15 Laptop                   : 389994.0
2. Hisense 43in Smart TV          : 272993.0
3. Lenovo Tab M10                 : 199992.0


In [9]:
def high_value(rows: list[dict], threshold: float) -> list[dict]:
    """Transactions whose revenue is at or above threshold (use a comprehension)."""
    return [# return row that meets condition
            row 
            # looping through rows
            for row in rows 
            # checking if total >= threshold
            if row["quantity"] * row["unit_price_kes"] >= threshold]


In [10]:

def high_value(rows: list[dict], threshold: float) -> list[dict]:
    """Transactions whose revenue is at or above threshold (use a comprehension)."""
    return [# return row that meets condition
            row 
            # looping through rows
            for row in rows 
            # checking if total >= threshold
            if row["quantity"] * row["unit_price_kes"] >= threshold]


threshold = 50000 
big_sales = high_value(clean, threshold)

print("Sales above the threshold")
for row in big_sales:
    revenue = row["quantity"] * row["unit_price_kes"]
    print(f"{row["sale_id"]}    {row["sale_date"]:<10} {row["store"]:<20}  {row["product"]} : {row["quantity"]:>5} * {row["unit_price_kes"]:>10,.2f} = {revenue:>10,.2f}"
          )


stores = sorted({row["store"] for row in rows})
big_sales_by_store = {store: sum(1 for row in big_sales if row["store"] == store) for store in stores}


print()
print("Sales over the threshold per store")
for store, count in big_sales_by_store.items():
    print(f"{store:<20} {count}")


Sales above the threshold
S-3013    2026-09-08 Westgate              Lenovo Tab M10 :     4 *  24,999.00 =  99,996.00
S-3017    2026-09-10 Kisumu Mega Plaza     Hisense 43in Smart TV :     4 *  38,999.00 = 155,996.00
S-3022    2026-09-13 Nairobi CBD           HP 15 Laptop :     4 *  64,999.00 = 259,996.00
S-3036    2026-09-21 Nakuru Westside       Tecno Spark 20 :     4 *  13,499.00 =  53,996.00
S-3048    2026-09-30 Kisumu Mega Plaza     HP 15 Laptop :     2 *  64,999.00 = 129,998.00

Sales over the threshold per store
Kisumu Mega Plaza    2
Mombasa City Mall    0
Nairobi CBD          1
Nakuru Westside      1
Westgate             1


In [11]:
# Reusable statistics - implement these yourself, no statistics module.
def mean(values: list[float]) -> float:
    # if no values passed
    if not values:
        raise ValueError("Need at least 1 value to find mean")
    # finding mean
    return sum(values)/ len(values)



In [12]:
def median(values: list[float]) -> float:
    if not values:
        raise ValueError("Need at least 1 value to find median")
    
    ordered = sorted(values)
    # if len of values is even
    mid = len(ordered) // 2
    # if len of values is odd, pick value at mid position as meadian
    if len(ordered) % 2 == 1:
        return ordered[mid]
    # get median by mean of values at the middle of dataset
    return (ordered[mid - 1] + ordered[mid]) / 2


In [13]:
def minimum_maximum(values: list[float]) -> tuple[float, float]:
    if not values:
        raise ValueError("Need at least 1 value to find min/max value")

    # set lowest, highest to 1st value in list
    lowest = highest = values[0]
    for value in values[1:]:
        if value < lowest:
            lowest = value
        if value > highest:
            highest = value
    return lowest, highest


In [14]:
# reveue per row

revenues = [row["quantity"] * row["unit_price_kes"]
            for row in clean]

low, high = minimum_maximum(revenues)
print(f"Transactions: {len(revenues)}")
print(f"Mean revenue:   KES {mean(revenues):,.2f}")
print(f"Median revenue: KES {median(revenues):,.2f}")
print(f"Smallest sale:  KES {low:,.2f}")
print(f"Largest sale:   KES {high:,.2f}")

Transactions: 46
Mean revenue:   KES 28,495.91
Median revenue: KES 11,097.00
Smallest sale:  KES 1,499.00
Largest sale:   KES 259,996.00


---

In [15]:
_NUMBER = r"(\d{1,3}(,\d{3})+|\d+)(\.\d+)?"

def _clean_numeric(series: pd.Series) -> pd.Series:
    s= (series.str.strip().str.strip("\"'").str.strip()
        .str.replace(r"(?i)^kes\s*", "", regex=True).str.strip())

    valid = s.str.fullmatch(_NUMBER)
    return pd.to_numeric(s.str.replace(",", "", regex=False).where(valid), errors="coerce")


In [18]:

def verify_with_pandas(path: Path, my_results: dict) -> None:
    """Recompute the headline numbers with pandas and print whether each one matches."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8")
    df["quantity"] = _clean_numeric(df["quantity"])
    df["unit_price_kes"] = _clean_numeric(df["unit_price_kes"])

    # skip rules: both numbers valid, quantity a whole number
    df = df[df["quantity"].notna() & (df["quantity"] % 1 == 0) & df["unit_price_kes"].notna()].copy()
    df["revenue"] = df["quantity"] * df["unit_price_kes"]

    expected = {
        "rows": len(df),
        "total": df["revenue"].sum(),
        "mean": df["revenue"].mean(),
        "median": df["revenue"].median(),
        "min": df["revenue"].min(),
        "max": df["revenue"].max(),
        "high_value_count": int((df["revenue"] >= 50_000).sum()),
        "by_product": df.groupby("product")["revenue"].sum().to_dict(),
        "by_store": df.groupby("store")["revenue"].sum().to_dict(),
        "by_payment_method": df.groupby("payment_method")["revenue"].sum().to_dict(),
    }

    def same(a, b) -> bool:
        if isinstance(a, dict) and isinstance(b, dict):
            return a.keys() == b.keys() and all(abs(a[k] - b[k]) < 0.005 for k in a)
        return abs(a - b) < 0.005

    mismatches = []
    for name, pandas_value in expected.items():
        mine = my_results.get(name)
        ok = mine is not None and same(mine, pandas_value)
        print(f"{'MATCH' if ok else 'MISMATCH':<9}{name}")
        if not ok:
            mismatches.append((name, mine, pandas_value))

    for name, mine, theirs in mismatches:
        print(f"  {name}: mine={mine!r}  pandas={theirs!r}")
    if not mismatches:
        print("\nAll checks match.")

In [19]:
revenues = [r["quantity"] * r["unit_price_kes"] for r in clean]
low, high = minimum_maximum(revenues)

my_results = {
    "rows": len(clean),
    "total": total_revenue(clean),
    "mean": mean(revenues),
    "median": median(revenues),
    "min": low,
    "max": high,
    "high_value_count": len(high_value(clean, 50_000)),
    "by_product": revenue_by(clean, "product"),
    "by_store": revenue_by(clean, "store"),
    "by_payment_method": revenue_by(clean, "payment_method"),
}

verify_with_pandas(DATA_PATH, my_results)

MATCH    rows
MATCH    total
MATCH    mean
MATCH    median
MATCH    min
MATCH    max
MATCH    high_value_count
MATCH    by_product
MATCH    by_store
MATCH    by_payment_method

All checks match.


---

In [20]:
def print_breakdown(title:str, totals: dict[str, float]) -> None:
    print(f" {title} (KES)")
    for key, value in sorted(totals.items(), key=lambda item: (-item[1], item[0])):
        print(f" {key:<20} {value:>20,.2f}")

In [ ]:
def main(path: Path = DATA_PATH) -> None:
    # 1. Read
    

In [ ]:

if __name__ == "__main__":
    # TODO
    pass
